# Этап 3: проверяем, что стрим положил в S3

Читаем все, что `stream_kafka_to_s3.py` записал в `s3a://raw/events/`, и проверяем:
сколько событий, как они разложились по датам и нет ли дублей.



In [ ]:
# Подключаем общую фабрику Spark-сессий из jobs/spark_session.py
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "jobs"))
from spark_session import get_spark

spark = get_spark("check-raw-events")

# Читаем все parquet-файлы стрима. Колонку event_date Spark восстановит из имен папок.
raw = spark.read.parquet("s3a://raw/events/")
# Регистрируем таблицу под именем raw_events, чтобы писать к ней SQL
raw.createOrReplaceTempView("raw_events")

raw.printSchema()

Ожидаем 10 колонок: поля события, `kafka_partition`, `kafka_offset` и `event_date`. У `event_ts` тип `timestamp`, а не `string`.

In [ ]:
# Сколько всего событий
spark.sql("SELECT count(*) AS events FROM raw_events").show()

Сравни с числом сообщений в топике `events` в Kafka UI. Пока стрим работает, они почти совпадают: разница - события последних 30 секунд.

In [ ]:
# Как события разложились по датам
spark.sql("""
    SELECT event_date, count(*) AS events
    FROM raw_events
    GROUP BY event_date
    ORDER BY event_date
""").show()

Ожидаем 3–4 даты: генератор раскидывает события по последним 3 дням. Это те самые партиции, по которым DAG в Airflow будет делать по запуску на день.

In [ ]:
# Первый взгляд на воронку: сколько каких событий
spark.sql("""
    SELECT event_type, count(*) AS events,
           round(100 * count(*) / sum(count(*)) OVER (), 1) AS pct
    FROM raw_events
    GROUP BY event_type
    ORDER BY events DESC
""").show()

Ожидаем примерно 70 / 20 / 10 % - веса из `events.py`.

In [ ]:
# Проверка на дубли: одно и то же событие не должно лежать в S3 дважды
spark.sql("""
    SELECT count(*) AS duplicated_events
    FROM (SELECT event_id FROM raw_events GROUP BY event_id HAVING count(*) > 1)
""").show()

Ожидаем 0 .

In [ ]:
spark.stop()